# 90 — Nodal full-node detection, picking, and shot-gather export

This notebook replaces the earlier detection-driven `90_*` workflow.

The key difference is:

> **Detections define event times. SDS extraction defines shot-gather contents. Picks never decide which stations are saved.**

The workflow is:

1. Discover all position-coded `DP*` stations in the SDS archive. This assumes the 1000 Hz `GP*` nodes have already been downsampled to 500 Hz and rewritten as `DP*` using notebook/script `86_*`.
2. Run network coincidence detection on `DPZ` only, using all available nodes in each named time window.
3. Merge duplicate detections across chunk boundaries.
4. For each detection, extract full 3-component `DPE/DPN/DPZ` shot gathers from SDS.
5. Write full-gather MiniSEED, component SEG-Y, and wiggle/image PNGs.
6. Run first-break autopicks and write pick rows.
7. Store the processing catalog in SQLite, with optional CSV exports for inspection.

Later notebooks can use the SQLite catalog to stack repeated nodal shots, compare nodal versus Geode/streamer gathers at common source positions, and build combined super-gathers.

## 1. Imports

This expects your project library layout to include `nodal_shotgather.py` and `segy_tools` under `../lib`, as in the earlier notebooks.

In [ ]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, asdict
import json
import sqlite3
import uuid
import traceback
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import Stream, UTCDateTime

# Local project library
import sys
LIB = Path("../lib").resolve()
if str(LIB) not in sys.path:
    sys.path.append(str(LIB))

from nodal_shotgather import (
    DetectionConfig,
    PickingConfig,
    read_deployment_from_sds,
    preprocess_for_detection,
    detect_network_events,
    preprocess_for_picking,
    pick_baer_aic_on_trace,
    try_ar_pick_short_station,
    consensus_pick_for_station,
)

# SEG-Y / plotting helpers. These should exist in ../lib/segy_tools.
try:
    from segy_tools.gather import gather_arrays_to_stream
    from segy_tools.io import write_segy
    from segy_tools.plotting import plot_wiggle_gather, plot_image_gather
    HAVE_SEGY_TOOLS = True
except Exception as e:
    HAVE_SEGY_TOOLS = False
    print("WARNING: could not import segy_tools helpers. SEG-Y/PNG export may be limited.")
    print(e)

## 2. Configuration

Update paths if needed. The SDS root should be the **position-coded SDS archive after running 86**, so all nodes appear as `DP*` at 500 Hz.

In [ ]:
# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
SDS_ROOT = PROJECT_ROOT / "nodal_sds_position_codes"

# New output root. This notebook does not overwrite the old detected-event tree.
OUT_ROOT = PROJECT_ROOT / "nodal_fullnode_shotgathers_v3"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"
CSV_EXPORT_DIR = OUT_ROOT / "catalog_exports"
CSV_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Optional metadata workbook for later source-position matching.
# Leave None to run detection/extraction only.
METADATA_WORKBOOK = Path("/Volumes/tachyon/LBSSP_DATA/jochen_field_notes_metadata_tables.xlsx")
if not METADATA_WORKBOOK.exists():
    METADATA_WORKBOOK = None

# -----------------------------------------------------------------------------
# Named processing windows.
# These are UTC SmartSolo/SDS times from the earlier 90_* notebook.
# -----------------------------------------------------------------------------
TIMEWINDOWS = {
    "T1_N1_Streamer": (UTCDateTime("2026-05-16T17:13:00"), UTCDateTime("2026-05-16T21:48:00")),
    "T1_N2_Nodal1": (UTCDateTime("2026-05-17T16:00:00"), UTCDateTime("2026-05-17T19:00:00")),
    "T1_N2_Refraction1m": (UTCDateTime("2026-05-18T16:03:54"), UTCDateTime("2026-05-18T18:38:33")),
    "T1_N2_Refraction2m": (UTCDateTime("2026-05-18T20:18:44"), UTCDateTime("2026-05-18T23:12:53")),
    "T1_N2_Nodal2": (UTCDateTime("2026-05-19T12:59:00"), UTCDateTime("2026-05-19T13:21:00")),
    "T1_N3_Nodal3": (UTCDateTime("2026-05-19T13:59:00"), UTCDateTime("2026-05-19T14:15:00")),
    "T3_N4_Refraction1am": (UTCDateTime("2026-05-19T16:02:00"), UTCDateTime("2026-05-19T18:25:00")),
}

# Leave as None to run all windows, or set a list for testing.
RUN_LABELS = None  # e.g., ["T1_N1_Streamer"] for testing

# -----------------------------------------------------------------------------
# Detection and extraction settings
# -----------------------------------------------------------------------------
DETECTION_CHANNEL = "DPZ"       # Z-only detection on all downsampled/real nodes
EXTRACTION_CHANNELS = ["DPE", "DPN", "DPZ"]
DETECTION_SAMPLE_RATE_HZ = 500.0

# Exclude known moving trigger/source node if it appears in SDS.
EXCLUDE_STATIONS = {"12806", "012806", "45012806"}

# Chunked detection avoids loading many hours of data at once.
CHUNK_SECONDS = 60.0
CHUNK_OVERLAP_SECONDS = 2.0

# Event extraction windows for full shot gathers.
# These are relative to detection on_time. Increase pre_s if first breaks are clipped.
GATHER_PRE_S = 0.10
GATHER_POST_S = 0.90

# Avoid detecting/keeping duplicate events on chunk overlaps or repeated trigger branches.
MERGE_TOLERANCE_S = 0.08
MIN_EVENT_SPACING_S = 0.20

# Development limits
MAX_EVENTS_PER_WINDOW = None  # e.g. 10 for testing
WRITE_MSEED = True
WRITE_SEGY = True
WRITE_PNG = True
MAKE_PICK_DIAGNOSTICS = False
EXPORT_CSV_SNAPSHOTS = True

# Detection thresholds. Tune after looking at the event catalog.
det_cfg = DetectionConfig(
    freqmin=5.0,
    freqmax=180.0,
    corners=4,
    zerophase=True,
    sta_seconds=0.02,
    lta_seconds=0.30,
    threshold_on=4.0,
    threshold_off=1.5,
    min_channels=10,   # all 36 nodes should now be available as DPZ; tune this
    min_snr=6.0,
    pretrigger_seconds=GATHER_PRE_S,
    posttrigger_seconds=GATHER_POST_S,
)

pick_cfg = PickingConfig(
    freqmin=10.0,
    freqmax=150.0,
    corners=4,
    zerophase=False,
    pick_tolerance_s=0.02,
    min_votes=2,
    include_ar_s=False,
    mute_seconds=0.02,
)

print("SDS_ROOT:", SDS_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("CATALOG_DB:", CATALOG_DB)
print("RUN_LABELS:", RUN_LABELS)

# Safety and metadata matching
MIN_FREE_GB = 50.0
METADATA_MATCH_TOLERANCE_S = 2.0
# Rerun policy: delete previous nodal rows and products from this output root before processing.
# Set False if you are deliberately appending a second nodal processing run to the same catalog.
REPLACE_EXISTING_NODAL_RUNS = True


In [ ]:
# ------------------------------------------------------------------
# Backup SQLite catalog before modifying it
# ------------------------------------------------------------------

from pathlib import Path
from datetime import datetime
import shutil

catalog_db = Path(CATALOG_DB)

if catalog_db.exists():

    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

    backup_db = (
        catalog_db.parent
        / f"{catalog_db.stem}_backup_{timestamp}{catalog_db.suffix}"
    )

    shutil.copy2(catalog_db, backup_db)

    print(f"Catalog backed up to:\n{backup_db}")

else:
    print(f"Catalog does not yet exist:\n{catalog_db}")


MAX_BACKUPS_TO_KEEP = 5

backups = sorted(
    catalog_db.parent.glob(
        f"{catalog_db.stem}_backup_*{catalog_db.suffix}"
    )
)

while len(backups) > MAX_BACKUPS_TO_KEEP:
    old = backups.pop(0)
    print(f"Removing old backup: {old.name}")
    old.unlink()

## 3. SQLite catalog schema

SQLite is the source of truth. CSVs are exported only as snapshots for inspection.

In [ ]:

def connect_catalog(db_path: Path) -> sqlite3.Connection:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("PRAGMA journal_mode = WAL")
    conn.execute("PRAGMA busy_timeout = 30000")
    return conn


def table_columns(conn: sqlite3.Connection, table: str) -> set[str]:
    try:
        return {row[1] for row in conn.execute(f"PRAGMA table_info({table})").fetchall()}
    except Exception:
        return set()


def ensure_columns(conn: sqlite3.Connection, table: str, columns: dict[str, str]):
    """Add missing columns to an existing SQLite table.

    This lets the nodal notebook share the same SQLite catalog first populated by
    the Geode metadata notebook, whose tables have a Geode-oriented schema.
    """
    existing = table_columns(conn, table)
    for name, decl in columns.items():
        if name not in existing:
            conn.execute(f"ALTER TABLE {table} ADD COLUMN {name} {decl}")
    conn.commit()


def init_catalog(conn: sqlite3.Connection):
    # Create tables if this notebook is run before 89_*. If 89_* already created
    # the tables, we add the nodal-specific columns below.
    conn.executescript(
    """
    CREATE TABLE IF NOT EXISTS processing_runs (
        run_id TEXT PRIMARY KEY,
        notebook_name TEXT,
        run_time_utc TEXT,
        input_sds_root TEXT,
        output_root TEXT,
        parameters_json TEXT,
        notes TEXT
    );

    CREATE TABLE IF NOT EXISTS receiver_geometry (
        geometry_id TEXT,
        event_id TEXT,
        instrument_system TEXT,
        line TEXT,
        transect TEXT,
        survey TEXT,
        receiver_index INTEGER,
        receiver_x_m REAL,
        receiver_y_m REAL,
        receiver_spacing_m REAL,
        component TEXT,
        sample_rate_hz REAL,
        geometry_status TEXT,
        geometry_note TEXT,
        run_id TEXT
    );

    CREATE TABLE IF NOT EXISTS shot_events (
        event_id TEXT PRIMARY KEY,
        instrument_system TEXT,
        line TEXT,
        transect TEXT,
        survey TEXT,
        survey_type TEXT,
        shot_no INTEGER,
        file_no INTEGER,
        source_x_m REAL,
        source_type TEXT,
        n_blows INTEGER,
        n_shots INTEGER,
        operator TEXT,
        plate_type TEXT,
        shot_time_local TEXT,
        shot_time_utc TEXT,
        receiver_first_m REAL,
        receiver_last_m REAL,
        receiver_spacing_m REAL,
        nominal_shot_spacing_m REAL,
        geode_read_ok INTEGER,
        geode_read_format TEXT,
        geode_n_traces INTEGER,
        geode_sampling_rate_hz REAL,
        geode_duration_s_first_trace REAL,
        geode_file_path TEXT,
        geode_folder TEXT,
        geode_match_status TEXT,
        geode_match_note TEXT,
        source_page TEXT,
        confidence TEXT,
        review_status TEXT,
        comment TEXT,
        metadata_source_sheet TEXT,
        extra_json TEXT,
        run_id TEXT
    );

    CREATE TABLE IF NOT EXISTS shot_gather_files (
        gather_file_id TEXT PRIMARY KEY,
        event_id TEXT,
        gather_id TEXT,
        instrument_system TEXT,
        line TEXT,
        transect TEXT,
        survey TEXT,
        component TEXT,
        file_type TEXT,
        file_path TEXT,
        source_file_no INTEGER,
        n_traces INTEGER,
        n_receivers INTEGER,
        sample_rate_hz REAL,
        duration_s REAL,
        processing_level TEXT,
        geometry_id TEXT,
        run_id TEXT
    );

    CREATE TABLE IF NOT EXISTS trace_index (
        trace_index_id TEXT PRIMARY KEY,
        event_id TEXT,
        gather_id TEXT,
        instrument_system TEXT,
        line TEXT,
        transect TEXT,
        survey TEXT,
        source_x_m REAL,
        receiver_index INTEGER,
        receiver_x_m REAL,
        offset_m REAL,
        component TEXT,
        file_path TEXT,
        trace_number_in_file INTEGER,
        sample_rate_hz REAL,
        npts INTEGER,
        amplitude_scale REAL,
        run_id TEXT
    );

    CREATE TABLE IF NOT EXISTS picks (
        run_id TEXT,
        event_id TEXT,
        gather_id TEXT,
        instrument_system TEXT,
        network TEXT,
        station TEXT,
        location TEXT,
        channel TEXT,
        component TEXT,
        receiver_x_m REAL,
        source_x_m REAL,
        offset_m REAL,
        pick_time_utc TEXT,
        pick_time_relative_s REAL,
        phase TEXT,
        picker TEXT,
        pick_quality TEXT,
        snr REAL,
        details_json TEXT
    );

    CREATE TABLE IF NOT EXISTS processing_errors (
        run_id TEXT,
        context TEXT,
        label TEXT,
        event_id TEXT,
        error_text TEXT,
        traceback TEXT,
        time_utc TEXT
    );
    """
    )

    ensure_columns(conn, "processing_runs", {
        "input_sds_root": "TEXT", "output_root": "TEXT", "input_metadata_xlsx": "TEXT",
        "input_geode_file_times_csv": "TEXT", "catalog_db": "TEXT", "parameters_json": "TEXT",
        "notes": "TEXT",
    })

    ensure_columns(conn, "receiver_geometry", {
        "network": "TEXT", "location": "TEXT", "station": "TEXT", "elevation_m": "REAL",
        "channel_family": "TEXT", "active_start_utc": "TEXT", "active_end_utc": "TEXT",
        "event_id": "TEXT", "instrument_system": "TEXT", "line": "TEXT", "transect": "TEXT",
        "survey": "TEXT", "receiver_index": "INTEGER", "receiver_x_m": "REAL", "receiver_y_m": "REAL",
        "receiver_spacing_m": "REAL", "component": "TEXT", "sample_rate_hz": "REAL",
        "geometry_status": "TEXT", "geometry_note": "TEXT", "run_id": "TEXT",
    })

    ensure_columns(conn, "shot_events", {
        "instrument_system": "TEXT", "line": "TEXT", "network": "TEXT", "location": "TEXT",
        "timewindow_label": "TEXT", "geometry_id": "TEXT", "detection_time_utc": "TEXT",
        "on_time_utc": "TEXT", "off_time_utc": "TEXT", "duration_s": "REAL",
        "source_x_m": "REAL", "source_type": "TEXT", "operator": "TEXT", "plate_type": "TEXT",
        "n_blows": "INTEGER", "n_shots": "INTEGER", "matched_metadata_event_id": "TEXT",
        "metadata_match_status": "TEXT", "metadata_match_time_error_s": "REAL",
        "detection_n_seed_ids": "INTEGER", "detection_n_stations": "INTEGER",
        "detection_seed_ids": "TEXT", "detection_stations": "TEXT", "snr_rms": "REAL",
        "n_receivers_extracted": "INTEGER", "n_traces_extracted": "INTEGER", "status": "TEXT",
        "notes": "TEXT", "survey": "TEXT", "survey_type": "TEXT", "shot_no": "INTEGER", "file_no": "INTEGER",
        "shot_time_utc": "TEXT", "metadata_source_sheet": "TEXT", "run_id": "TEXT",
    })

    ensure_columns(conn, "shot_gather_files", {
        "gather_file_id": "TEXT", "event_id": "TEXT", "gather_id": "TEXT", "instrument_system": "TEXT",
        "line": "TEXT", "network": "TEXT", "location": "TEXT", "transect": "TEXT", "survey": "TEXT",
        "timewindow_label": "TEXT", "component": "TEXT", "file_type": "TEXT", "file_path": "TEXT",
        "source_file_no": "INTEGER", "n_traces": "INTEGER", "n_receivers": "INTEGER",
        "sample_rate_hz": "REAL", "duration_s": "REAL", "pre_s": "REAL", "post_s": "REAL",
        "processing_level": "TEXT", "geometry_id": "TEXT", "run_id": "TEXT",
    })

    ensure_columns(conn, "trace_index", {
        "trace_index_id": "TEXT", "event_id": "TEXT", "gather_id": "TEXT", "instrument_system": "TEXT",
        "line": "TEXT", "network": "TEXT", "station": "TEXT", "location": "TEXT", "channel": "TEXT",
        "component": "TEXT", "source_x_m": "REAL", "receiver_index": "INTEGER", "receiver_x_m": "REAL",
        "offset_m": "REAL", "starttime_utc": "TEXT", "file_path": "TEXT", "trace_number_in_file": "INTEGER",
        "trace_index": "INTEGER", "sample_rate_hz": "REAL", "sampling_rate_hz": "REAL", "npts": "INTEGER",
        "amplitude_scale": "REAL", "run_id": "TEXT",
    })

    conn.commit()


RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
conn = connect_catalog(CATALOG_DB)
init_catalog(conn)

if REPLACE_EXISTING_NODAL_RUNS:
    # Remove prior nodal catalog rows. Output files are not deleted automatically;
    # using a new OUT_ROOT version is safer for reruns.
    for sql in [
        "DELETE FROM picks WHERE instrument_system = 'nodal'",
        "DELETE FROM trace_index WHERE instrument_system = 'nodal'",
        "DELETE FROM shot_gather_files WHERE instrument_system = 'nodal'",
        "DELETE FROM receiver_geometry WHERE instrument_system = 'nodal'",
        "DELETE FROM shot_events WHERE instrument_system = 'nodal'",
    ]:
        try:
            conn.execute(sql)
        except Exception as e:
            print(f"Nodal cleanup skipped/failed for {sql}: {e}")
    conn.commit()
    print("Removed previous nodal catalog rows before this run.")

params = {
    "SDS_ROOT": str(SDS_ROOT),
    "OUT_ROOT": str(OUT_ROOT),
    "CATALOG_DB": str(CATALOG_DB),
    "TIMEWINDOWS": {k: [str(v[0]), str(v[1])] for k, v in TIMEWINDOWS.items()},
    "RUN_LABELS": RUN_LABELS,
    "DETECTION_CHANNEL": DETECTION_CHANNEL,
    "EXTRACTION_CHANNELS": EXTRACTION_CHANNELS,
    "CHUNK_SECONDS": CHUNK_SECONDS,
    "CHUNK_OVERLAP_SECONDS": CHUNK_OVERLAP_SECONDS,
    "GATHER_PRE_S": GATHER_PRE_S,
    "GATHER_POST_S": GATHER_POST_S,
    "MERGE_TOLERANCE_S": MERGE_TOLERANCE_S,
    "MIN_EVENT_SPACING_S": MIN_EVENT_SPACING_S,
    "MIN_FREE_GB": MIN_FREE_GB,
    "METADATA_MATCH_TOLERANCE_S": METADATA_MATCH_TOLERANCE_S,
    "REPLACE_EXISTING_NODAL_RUNS": REPLACE_EXISTING_NODAL_RUNS,
    "det_cfg": asdict(det_cfg),
    "pick_cfg": asdict(pick_cfg),
}

# Insert only columns that exist in processing_runs, since 89_* and 90_* may
# have slightly different run-provenance schemas.
run_row = {
    "run_id": RUN_ID,
    "notebook_name": "90_nodal_fullnode_detection_pick_and_shotgathers.ipynb",
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "input_sds_root": str(SDS_ROOT),
    "output_root": str(OUT_ROOT),
    "catalog_db": str(CATALOG_DB),
    "parameters_json": json.dumps(params, indent=2),
    "notes": "Full-node nodal shot-gather factory; DPZ detection, DP[ENZ] extraction, metadata matching to Geode catalog.",
}
cols = list(table_columns(conn, "processing_runs") & set(run_row.keys()))
pd.DataFrame([{k: run_row[k] for k in cols}]).to_sql("processing_runs", conn, if_exists="append", index=False)
conn.commit()
print("RUN_ID:", RUN_ID)


## 4. Utility functions

Position-coded stations are interpreted as centimetres along the line, e.g. `08808 -> 88.08 m`.

In [ ]:
def parse_label(label: str) -> tuple[str, str]:
    """Parse labels like T1_N1_Streamer into network/location."""
    parts = str(label).split("_")
    if len(parts) < 2:
        raise ValueError(f"Cannot parse network/location from label: {label}")
    return parts[0], parts[1]


def station_to_x_m(station: str) -> float:
    """Position-coded station name -> x position in metres."""
    s = str(station).strip()
    if not s.isdigit():
        return np.nan
    return int(s) / 100.0


def channel_to_component(channel: str) -> str:
    return str(channel)[-1].upper()


def stream_stations(st: Stream) -> list[str]:
    return sorted({str(tr.stats.station) for tr in st})


def stream_seed_ids(st: Stream) -> list[str]:
    return sorted({tr.id for tr in st})


def discover_sds_stations(
    sds_root: Path,
    network: str,
    location: str,
    channel_pattern: str = "DPZ",
    exclude_stations: set[str] | None = None,
) -> list[str]:
    """Discover station codes with files matching an SDS channel selector."""
    exclude_stations = exclude_stations or set()
    root = Path(sds_root)
    stations = set()
    # Pattern: YEAR/NET/STA/CHAN.D/NET.STA.LOC.CHAN.D.YEAR.JDAY
    for p in root.rglob(f"{network}.*.{location}.{channel_pattern}.D.*.*"):
        if p.name.startswith("._"):
            continue
        try:
            parts = p.name.split(".")
            if len(parts) >= 7:
                net, sta, loc, cha = parts[:4]
                if net == network and loc == location:
                    if sta not in exclude_stations:
                        stations.add(sta)
        except Exception:
            pass
    return sorted(stations, key=lambda s: (station_to_x_m(s), s))


def build_geometry_df(label: str, start: UTCDateTime, end: UTCDateTime, stations: list[str]) -> pd.DataFrame:
    network, location = parse_label(label)
    rows = []
    for sta in stations:
        rows.append({
            "run_id": RUN_ID,
            "geometry_id": f"{label}_{network}_{location}_DPall",
            "instrument_system": "nodal",
            "line": network,
            "network": network,
            "location": location,
            "station": sta,
            "receiver_x_m": station_to_x_m(sta),
            "receiver_y_m": 0.0,
            "elevation_m": 0.0,
            "channel_family": "DP",
            "sample_rate_hz": DETECTION_SAMPLE_RATE_HZ,
            "active_start_utc": str(start),
            "active_end_utc": str(end),
        })
    return pd.DataFrame(rows)


def utc_to_str(x) -> str | None:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    try:
        return str(UTCDateTime(x))
    except Exception:
        return str(x)


def log_error(conn, context, label=None, event_id=None, exc=None):
    conn.execute(
        "INSERT INTO processing_errors VALUES (?, ?, ?, ?, ?, ?, ?)",
        (
            RUN_ID,
            context,
            label,
            event_id,
            str(exc),
            traceback.format_exc(),
            datetime.now(timezone.utc).isoformat(),
        ),
    )
    conn.commit()


def check_free_space(path: Path, min_free_gb: float = MIN_FREE_GB):
    import shutil
    usage = shutil.disk_usage(Path(path))
    free_gb = usage.free / (1024**3)
    if free_gb < min_free_gb:
        raise RuntimeError(f"Only {free_gb:.1f} GiB free at {path}; stopping before writing more outputs.")
    return free_gb


## 5. Detection helpers

Detection is run on `DPZ` only, after the `86_*` downsampling step has converted `GP*` data to `DP*` channels. The resulting detection stream should include both original 500 Hz nodes and downsampled 1000 Hz nodes.

In [ ]:
def iter_time_chunks(start: UTCDateTime, end: UTCDateTime, chunk_s: float, overlap_s: float):
    t = UTCDateTime(start)
    i = 0
    while t < end:
        core_start = t
        core_end = min(t + chunk_s, end)
        read_start = max(start, core_start - overlap_s)
        read_end = min(end, core_end + overlap_s)
        yield i, core_start, core_end, read_start, read_end
        t = core_end
        i += 1


def read_detection_stream(label: str, read_start: UTCDateTime, read_end: UTCDateTime, stations: list[str]) -> Stream:
    network, location = parse_label(label)
    # Read station='*' first because EnhancedSDSClient supports wildcards.
    st = read_deployment_from_sds(
        SDS_ROOT,
        network=network,
        location=location,
        station="*",
        channel=DETECTION_CHANNEL,
        starttime=read_start,
        endtime=read_end,
        merge=True,
        verbose=False,
    )
    # Keep only expected/excluded-filtered stations.
    keep = set(stations)
    st = Stream([tr for tr in st if str(tr.stats.station) in keep])
    if len(st):
        st.sort(keys=["station", "channel"])
    return st


def run_detection_for_window(label: str, start: UTCDateTime, end: UTCDateTime, stations: list[str]) -> pd.DataFrame:
    rows = []
    chunk_out = OUT_ROOT / label / "tables" / "chunk_detections"
    chunk_out.mkdir(parents=True, exist_ok=True)

    for chunk_index, core_start, core_end, read_start, read_end in iter_time_chunks(start, end, CHUNK_SECONDS, CHUNK_OVERLAP_SECONDS):
        print(f"{label} chunk {chunk_index}: {core_start} to {core_end}")
        try:
            st_raw = read_detection_stream(label, read_start, read_end, stations)
            if len(st_raw) == 0:
                print("  no traces")
                continue
            print(f"  read {len(st_raw)} traces, {len(stream_stations(st_raw))} stations")

            st_det = preprocess_for_detection(st_raw, det_cfg)
            df = detect_network_events(st_det, det_cfg, write_mseed=False, make_plots=False)
            if len(df) == 0:
                continue

            # Keep detections whose on_time lies inside the core chunk, avoiding duplicate overlap products.
            df = df.copy()
            df["on_time_dt"] = df["on_time"].apply(lambda x: UTCDateTime(x))
            df = df[(df["on_time_dt"] >= core_start) & (df["on_time_dt"] < core_end)].copy()
            if len(df) == 0:
                continue

            df["run_id"] = RUN_ID
            df["timewindow_label"] = label
            network, location = parse_label(label)
            df["network"] = network
            df["location"] = location
            df["chunk_index"] = chunk_index
            df["core_start_utc"] = str(core_start)
            df["core_end_utc"] = str(core_end)
            df["read_start_utc"] = str(read_start)
            df["read_end_utc"] = str(read_end)
            df["n_available_detection_stations"] = len(stations)
            df = df.drop(columns=["on_time_dt"])

            outcsv = chunk_out / f"{label}_{network}_{location}_{chunk_index:06d}_{core_start.strftime('%Y%m%dT%H%M%S')}_{core_end.strftime('%Y%m%dT%H%M%S')}_detections.csv"
            df.to_csv(outcsv, index=False)
            rows.append(df)
            print(f"  detections: {len(df)}")
        except Exception as e:
            print(f"  ERROR: {e}")
            log_error(conn, "detection_chunk", label=label, exc=e)

    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    return out


def merge_detection_catalog(df: pd.DataFrame, tolerance_s: float = 0.08, min_spacing_s: float = 0.20) -> pd.DataFrame:
    """Merge near-duplicate detections, keeping the row with largest n_seed_ids/SNR."""
    if len(df) == 0:
        return df.copy()

    d = df.copy()
    d["_on"] = d["on_time"].apply(lambda x: UTCDateTime(x).timestamp)
    d = d.sort_values("_on").reset_index(drop=True)

    groups = []
    current = [0]
    for i in range(1, len(d)):
        if d.loc[i, "_on"] - d.loc[current[-1], "_on"] <= tolerance_s:
            current.append(i)
        else:
            groups.append(current)
            current = [i]
    groups.append(current)

    keep_rows = []
    for g in groups:
        sub = d.loc[g].copy()
        # Prefer more triggering channels/stations, then higher SNR if present.
        sort_cols = []
        ascending = []
        for c in ["n_seed_ids", "n_stations", "snr_rms"]:
            if c in sub.columns:
                sort_cols.append(c)
                ascending.append(False)
        if sort_cols:
            sub = sub.sort_values(sort_cols, ascending=ascending)
        keep_rows.append(sub.iloc[0])

    merged = pd.DataFrame(keep_rows).drop(columns=["_on"], errors="ignore").reset_index(drop=True)

    # Enforce minimum spacing by keeping stronger row when detections are still too close.
    if len(merged) <= 1:
        return merged
    merged["_on"] = merged["on_time"].apply(lambda x: UTCDateTime(x).timestamp)
    final = []
    for _, row in merged.sort_values("_on").iterrows():
        if not final:
            final.append(row)
            continue
        if row["_on"] - final[-1]["_on"] < min_spacing_s:
            prev = final[-1]
            score_prev = float(prev.get("n_seed_ids", 0) or 0) + 0.01 * float(prev.get("snr_rms", 0) or 0)
            score_new = float(row.get("n_seed_ids", 0) or 0) + 0.01 * float(row.get("snr_rms", 0) or 0)
            if score_new > score_prev:
                final[-1] = row
        else:
            final.append(row)
    return pd.DataFrame(final).drop(columns=["_on"], errors="ignore").reset_index(drop=True)

## 6. Full-gather extraction, plotting, SEG-Y export, and picking

SEG-Y and plot geometry are based directly on position-coded station names, not trace index.

In [ ]:
def read_full_gather_from_sds(label: str, event_time: UTCDateTime, stations: list[str]) -> Stream:
    network, location = parse_label(label)
    t1 = event_time - GATHER_PRE_S
    t2 = event_time + GATHER_POST_S
    st = read_deployment_from_sds(
        SDS_ROOT,
        network=network,
        location=location,
        station="*",
        channel="DP?",
        starttime=t1,
        endtime=t2,
        merge=True,
        verbose=False,
    )
    keep = set(stations)
    st = Stream([tr for tr in st if str(tr.stats.station) in keep and tr.stats.channel in EXTRACTION_CHANNELS])
    st.sort(keys=["station", "channel"])
    return st


def stream_to_component_arrays(st: Stream, component: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, list]:
    """Return time, data(ntr,npts), receiver_x_m, ordered traces for one component."""
    component = component.upper()
    traces = [tr for tr in st if channel_to_component(tr.stats.channel) == component]
    traces = sorted(traces, key=lambda tr: (station_to_x_m(tr.stats.station), tr.stats.station))
    if not traces:
        return np.array([]), np.empty((0, 0)), np.array([]), []
    npts = min(tr.stats.npts for tr in traces)
    dt = float(traces[0].stats.delta)
    time = np.arange(npts) * dt
    data = np.vstack([np.asarray(tr.data[:npts], dtype=np.float32) for tr in traces])
    rx = np.asarray([station_to_x_m(tr.stats.station) for tr in traces], dtype=float)
    return time, data, rx, traces


def write_component_products(
    st_event: Stream,
    label: str,
    event_id: str,
    gather_id: str,
    source_x_m: float | None,
) -> tuple[list[dict], list[dict]]:
    """Write MiniSEED, component SEG-Y and PNG files. Return file rows and trace-index rows."""
    network, location = parse_label(label)
    event_dir = OUT_ROOT / label
    mseed_dir = event_dir / "gathers_mseed"
    segy_dir = event_dir / "gathers_segy"
    fig_dir = event_dir / "figures"
    for d in [mseed_dir, segy_dir, fig_dir]:
        d.mkdir(parents=True, exist_ok=True)

    file_rows = []
    trace_rows = []
    source_x_for_headers = 0.0 if source_x_m is None or not np.isfinite(source_x_m) else float(source_x_m)

    # Full 3C MiniSEED
    mseed_path = mseed_dir / f"{event_id}_DPall.mseed"
    if WRITE_MSEED:
        st_event.write(str(mseed_path), format="MSEED")
        file_rows.append({
            "gather_file_id": f"GF_{event_id}_MSEED_3C",
            "run_id": RUN_ID, "event_id": event_id, "gather_id": gather_id,
            "line": network, "network": network, "location": location, "timewindow_label": label,
            "instrument_system": "nodal", "component": "3C", "file_type": "mseed",
            "file_path": str(mseed_path), "n_traces": len(st_event),
            "n_receivers": len(stream_stations(st_event)),
            "sample_rate_hz": float(st_event[0].stats.sampling_rate) if len(st_event) else np.nan,
            "pre_s": GATHER_PRE_S, "post_s": GATHER_POST_S,
            "processing_level": "raw_fullnode",
        })

    # Trace index points to the MiniSEED bundle by default.
    for i, tr in enumerate(sorted(st_event, key=lambda tr: (station_to_x_m(tr.stats.station), tr.stats.channel))):
        rx = station_to_x_m(tr.stats.station)
        sx = np.nan if source_x_m is None else source_x_m
        trace_rows.append({
            "trace_index_id": f"TR_{event_id}_{i:05d}",
            "run_id": RUN_ID, "event_id": event_id, "gather_id": gather_id,
            "line": network,
            "instrument_system": "nodal", "network": tr.stats.network,
            "station": tr.stats.station, "location": tr.stats.location,
            "channel": tr.stats.channel, "component": channel_to_component(tr.stats.channel),
            "receiver_x_m": rx, "source_x_m": sx,
            "offset_m": np.nan if not np.isfinite(sx) else rx - sx,
            "starttime_utc": str(tr.stats.starttime),
            "sampling_rate_hz": float(tr.stats.sampling_rate), "npts": int(tr.stats.npts),
            "file_path": str(mseed_path), "trace_index": i,
            "amplitude_scale": 1.0,
        })

    # Component SEG-Y and figures
    for comp in ["Z", "N", "E"]:
        time, data, rx, traces = stream_to_component_arrays(st_event, comp)
        if len(traces) == 0:
            continue
        dt = float(traces[0].stats.delta)
        sr = float(traces[0].stats.sampling_rate)

        if WRITE_SEGY and HAVE_SEGY_TOOLS:
            segy_path = segy_dir / comp / f"{event_id}_{comp}.sgy"
            segy_path.parent.mkdir(parents=True, exist_ok=True)
            segy_st = gather_arrays_to_stream(
                data=data,
                dt_s=dt,
                starttime=traces[0].stats.starttime,
                receiver_x_m=rx,
                source_x_m=source_x_for_headers,
                shot_number=int(event_id.split("E")[-1]) if "E" in event_id else 1,
                station_prefix="R",
                network=network,
                component=comp,
            )
            write_segy(segy_st, segy_path)
            file_rows.append({
                "gather_file_id": f"GF_{event_id}_{comp}_SEGY",
                "run_id": RUN_ID, "event_id": event_id, "gather_id": f"{gather_id}_{comp}",
                "line": network, "network": network, "location": location, "timewindow_label": label,
                "instrument_system": "nodal", "component": comp, "file_type": "segy",
                "file_path": str(segy_path), "n_traces": int(data.shape[0]),
                "n_receivers": int(len(rx)), "sample_rate_hz": sr,
                "pre_s": GATHER_PRE_S, "post_s": GATHER_POST_S,
                "processing_level": "raw_fullnode_component",
            })

        if WRITE_PNG and HAVE_SEGY_TOOLS:
            wiggle_path = fig_dir / f"wiggle_{comp}" / f"{event_id}_{comp}_wiggle.png"
            image_path = fig_dir / f"image_{comp}" / f"{event_id}_{comp}_image.png"
            title = f"{event_id} {comp} — {label}"
            plot_wiggle_gather(
                time, data, rx,
                source_x_m=source_x_m,
                title=title,
                tmin=0.0, tmax=min(GATHER_PRE_S + GATHER_POST_S, 1.0),
                scale=0.8, clip_percentile=99, normalize=True,
                outfile=wiggle_path,
            )
            plot_image_gather(
                time, data, rx,
                source_x_m=source_x_m,
                title=title,
                tmin=0.0, tmax=min(GATHER_PRE_S + GATHER_POST_S, 1.0),
                clip_percentile=98,
                outfile=image_path,
            )
            for path, ftype in [(wiggle_path, "png_wiggle"), (image_path, "png_image")]:
                file_rows.append({
                    "gather_file_id": f"GF_{event_id}_{comp}_{ftype.upper()}",
                    "run_id": RUN_ID, "event_id": event_id, "gather_id": f"{gather_id}_{comp}",
                    "line": network, "network": network, "location": location, "timewindow_label": label,
                    "instrument_system": "nodal", "component": comp, "file_type": ftype,
                    "file_path": str(path), "n_traces": int(data.shape[0]),
                    "n_receivers": int(len(rx)), "sample_rate_hz": sr,
                    "pre_s": GATHER_PRE_S, "post_s": GATHER_POST_S,
                    "processing_level": "raw_fullnode_component",
                })

    return file_rows, trace_rows


def pick_event_stream(st_event: Stream, event_id: str, gather_id: str, source_x_m: float | None) -> pd.DataFrame:
    """Run GeoPark-style component pickers and station consensus on one full shot gather."""
    rows = []
    if len(st_event) == 0:
        return pd.DataFrame()

    st_pick = preprocess_for_picking(st_event.copy(), pick_cfg)
    event_start = min(tr.stats.starttime for tr in st_pick)

    for station in stream_stations(st_pick):
        st_sta = st_pick.select(station=station)
        if len(st_sta) == 0:
            continue
        picks_by_comp = {}
        tr_z = tr_n = tr_e = None
        for comp in ["Z", "N", "E"]:
            sel = [tr for tr in st_sta if channel_to_component(tr.stats.channel) == comp]
            if not sel:
                continue
            tr = sel[0]
            if comp == "Z": tr_z = tr
            if comp == "N": tr_n = tr
            if comp == "E": tr_e = tr
            try:
                picks_by_comp[comp] = pick_baer_aic_on_trace(tr)
            except Exception as e:
                picks_by_comp[comp] = {"aic_ok": False, "baer_ok": False, "aic_error": str(e), "baer_error": str(e)}

        p_ar = s_ar = None
        if pick_cfg.include_ar_s and tr_z is not None and tr_n is not None and tr_e is not None:
            try:
                p_ar, s_ar = try_ar_pick_short_station(tr_z, tr_n, tr_e, f1=pick_cfg.freqmin, f2=pick_cfg.freqmax)
            except Exception:
                pass

        if tr_z is not None and picks_by_comp:
            consensus = consensus_pick_for_station(
                picks_by_comp,
                tr_z,
                p_ar=p_ar,
                s_ar=s_ar,
                pick_tolerance_s=pick_cfg.pick_tolerance_s,
                min_votes=pick_cfg.min_votes,
                min_weight=pick_cfg.min_weight,
                include_ar_s=pick_cfg.include_ar_s,
                baer_weight=pick_cfg.baer_weight,
                mute_seconds=pick_cfg.mute_seconds,
            )
        else:
            consensus = {"ok": False, "time": None, "relative_s": np.nan, "n_votes": 0, "weight": 0, "methods": [], "components": []}

        rx = station_to_x_m(station)
        sx = np.nan if source_x_m is None else source_x_m

        # Individual picker rows
        for comp, d in picks_by_comp.items():
            tr = [tr for tr in st_sta if channel_to_component(tr.stats.channel) == comp][0]
            for method in ["aic", "baer"]:
                ok = bool(d.get(f"{method}_ok", False))
                t = d.get(f"{method}_time") if ok else None
                rows.append({
                    "run_id": RUN_ID, "event_id": event_id, "gather_id": gather_id,
                    "instrument_system": "nodal", "network": tr.stats.network,
                    "station": station, "location": tr.stats.location,
                    "channel": tr.stats.channel, "component": comp,
                    "receiver_x_m": rx, "source_x_m": sx,
                    "offset_m": np.nan if not np.isfinite(sx) else rx - sx,
                    "pick_time_utc": str(t) if t is not None else None,
                    "pick_time_relative_s": float(t - event_start) if t is not None else np.nan,
                    "phase": "P?", "picker": method,
                    "pick_quality": "candidate" if ok else "failed",
                    "snr": np.nan,
                    "details_json": json.dumps({k: str(v) for k, v in d.items()}),
                })

        # Consensus row
        if consensus.get("ok"):
            t = consensus.get("time")
            pick_time_utc = str(t)
            rel = float(t - event_start)
            quality = "consensus"
        else:
            pick_time_utc = None
            rel = np.nan
            quality = "failed"
        rows.append({
            "run_id": RUN_ID, "event_id": event_id, "gather_id": gather_id,
            "instrument_system": "nodal", "network": st_sta[0].stats.network,
            "station": station, "location": st_sta[0].stats.location,
            "channel": "DP?", "component": "3C",
            "receiver_x_m": rx, "source_x_m": sx,
            "offset_m": np.nan if not np.isfinite(sx) else rx - sx,
            "pick_time_utc": pick_time_utc,
            "pick_time_relative_s": rel,
            "phase": "P?", "picker": "consensus",
            "pick_quality": quality,
            "snr": np.nan,
            "details_json": json.dumps({
                "votes": consensus.get("n_votes"),
                "weight": consensus.get("weight"),
                "methods": consensus.get("methods"),
                "components": consensus.get("components"),
                "p_ar_s": p_ar,
                "s_ar_s": s_ar,
            }),
        })
    return pd.DataFrame(rows)

## 7. Optional metadata matching hook

This is intentionally conservative. If no reliable metadata match is found, the event is still written with `metadata_match_status='unmatched_detection'` and `source_x_m=NULL`.

You can later improve this function using Jochen's spreadsheet columns once the shot-time table is finalized.

In [ ]:

# Load Geode/field metadata already indexed by 89_* and use it to annotate nodal detections.
# The matching is deliberately conservative: same canonical line and nearest shot_time_utc
# within METADATA_MATCH_TOLERANCE_S. Only Geode rows with both time and source_x_m are used.

def load_indexed_metadata_events(conn: sqlite3.Connection) -> pd.DataFrame:
    try:
        df = pd.read_sql(
            """
            SELECT event_id, instrument_system, line, transect, survey, survey_type,
                   shot_no, file_no, source_x_m, source_type, n_blows, n_shots,
                   operator, plate_type, shot_time_utc, metadata_source_sheet,
                   comment, confidence, review_status
            FROM shot_events
            WHERE instrument_system = 'geode'
              AND shot_time_utc IS NOT NULL
              AND source_x_m IS NOT NULL
            """,
            conn,
        )
    except Exception as e:
        print("Could not load indexed Geode metadata:", e)
        return pd.DataFrame()

    if len(df):
        df["shot_time_dt"] = pd.to_datetime(df["shot_time_utc"], utc=True, errors="coerce")
        df = df.dropna(subset=["shot_time_dt"])
        df["line"] = df["line"].astype(str)
    return df


metadata_events = load_indexed_metadata_events(conn)
print("metadata events available for matching:", len(metadata_events))
if len(metadata_events):
    display(metadata_events.groupby(["line", "survey", "survey_type"]).size().reset_index(name="n").head(20))


def match_metadata_for_detection(label: str, detection_time: UTCDateTime) -> dict:
    """Match a nodal detection to the nearest indexed Geode/field shot metadata row."""
    network, location = parse_label(label)
    if len(metadata_events) == 0:
        return {
            "source_x_m": None,
            "source_type": None,
            "operator": None,
            "plate_type": None,
            "n_blows": None,
            "n_shots": None,
            "matched_metadata_event_id": None,
            "metadata_match_status": "unmatched_no_metadata_events",
            "metadata_match_time_error_s": None,
            "survey": None,
            "survey_type": None,
            "shot_no": None,
            "file_no": None,
            "notes": None,
        }

    det_dt = pd.to_datetime(str(detection_time), utc=True)
    candidates = metadata_events[metadata_events["line"] == network].copy()
    if len(candidates) == 0:
        return {
            "source_x_m": None,
            "source_type": None,
            "operator": None,
            "plate_type": None,
            "n_blows": None,
            "n_shots": None,
            "matched_metadata_event_id": None,
            "metadata_match_status": "unmatched_no_same_line_metadata",
            "metadata_match_time_error_s": None,
            "survey": None,
            "survey_type": None,
            "shot_no": None,
            "file_no": None,
            "notes": None,
        }

    candidates["dt_abs_s"] = (candidates["shot_time_dt"] - det_dt).dt.total_seconds().abs()
    best = candidates.sort_values("dt_abs_s").iloc[0]
    dt_abs = float(best["dt_abs_s"])

    if dt_abs > METADATA_MATCH_TOLERANCE_S:
        return {
            "source_x_m": None,
            "source_type": None,
            "operator": None,
            "plate_type": None,
            "n_blows": None,
            "n_shots": None,
            "matched_metadata_event_id": None,
            "metadata_match_status": f"unmatched_nearest_{dt_abs:.3f}s_exceeds_tolerance",
            "metadata_match_time_error_s": dt_abs,
            "survey": None,
            "survey_type": None,
            "shot_no": None,
            "file_no": None,
            "notes": None,
        }

    return {
        "source_x_m": float(best["source_x_m"]),
        "source_type": best.get("source_type"),
        "operator": best.get("operator"),
        "plate_type": best.get("plate_type"),
        "n_blows": None if pd.isna(best.get("n_blows")) else int(best.get("n_blows")),
        "n_shots": None if pd.isna(best.get("n_shots")) else int(best.get("n_shots")),
        "matched_metadata_event_id": best.get("event_id"),
        "metadata_match_status": "matched_geode_metadata_by_time",
        "metadata_match_time_error_s": dt_abs,
        "survey": best.get("survey"),
        "survey_type": best.get("survey_type"),
        "shot_no": None if pd.isna(best.get("shot_no")) else int(best.get("shot_no")),
        "file_no": None if pd.isna(best.get("file_no")) else int(best.get("file_no")),
        "notes": best.get("comment"),
    }


## 8. Main processing loop

This is the expensive cell. Start with `RUN_LABELS = ['T1_N1_Streamer']` and `MAX_EVENTS_PER_WINDOW = 5` if testing.

In [ ]:
labels_to_run = list(TIMEWINDOWS.keys()) if RUN_LABELS is None else list(RUN_LABELS)
all_window_catalogs = []

for label in labels_to_run:
    check_free_space(OUT_ROOT, MIN_FREE_GB)
    print("\n" + "="*100)
    print("Processing", label)
    start, end = TIMEWINDOWS[label]
    network, location = parse_label(label)
    geometry_id = f"{label}_{network}_{location}_DPall"

    label_out = OUT_ROOT / label
    (label_out / "tables").mkdir(parents=True, exist_ok=True)

    # 1. Discover all DPZ stations for this network/location after 86_downsample.
    stations = discover_sds_stations(SDS_ROOT, network, location, DETECTION_CHANNEL, EXCLUDE_STATIONS)
    print(f"Discovered {len(stations)} {network}.{location}.{DETECTION_CHANNEL} stations")
    print(stations)
    if len(stations) == 0:
        continue

    # 2. Write receiver geometry to SQLite.
    geom_df = build_geometry_df(label, start, end, stations)
    geom_df.to_sql("receiver_geometry", conn, if_exists="append", index=False)
    geom_df.to_csv(label_out / "tables" / f"{label}_receiver_geometry.csv", index=False)

    # 3. Run chunked DPZ detection.
    raw_det = run_detection_for_window(label, start, end, stations)
    if len(raw_det) == 0:
        print("No detections")
        continue
    raw_path = label_out / "tables" / f"{label}_detected_events_raw_chunked.csv"
    raw_det.to_csv(raw_path, index=False)

    merged_det = merge_detection_catalog(raw_det, tolerance_s=MERGE_TOLERANCE_S, min_spacing_s=MIN_EVENT_SPACING_S)
    merged_path = label_out / "tables" / f"{label}_detected_events_merged.csv"
    merged_det.to_csv(merged_path, index=False)
    print(f"Raw detections: {len(raw_det)}; merged detections: {len(merged_det)}")

    if MAX_EVENTS_PER_WINDOW is not None:
        merged_det = merged_det.iloc[:MAX_EVENTS_PER_WINDOW].copy()

    # 4. For each detection, go back to SDS and extract all DP[ENZ] stations.
    for local_i, row in merged_det.reset_index(drop=True).iterrows():
        event_num = local_i + 1
        event_id = f"{label}_{network}_{location}_E{event_num:05d}"
        gather_id = f"{event_id}_nodal_DPall"
        print(f"\n{event_id}: on_time={row['on_time']}")

        try:
            event_time = UTCDateTime(row["on_time"])
            meta = match_metadata_for_detection(label, event_time)
            source_x_m = meta.get("source_x_m")

            st_event = read_full_gather_from_sds(label, event_time, stations)
            n_receivers = len(stream_stations(st_event))
            n_traces = len(st_event)
            print(f"  extracted {n_traces} traces, {n_receivers} receivers")

            # If extraction fails, still catalog the detected event.
            status = "ok" if len(st_event) else "no_traces_extracted"

            event_row = {
                "run_id": RUN_ID,
                "event_id": event_id,
                "instrument_system": "nodal",
                "line": network,
                "network": network,
                "location": location,
                "timewindow_label": label,
                "geometry_id": geometry_id,
                "detection_time_utc": utc_to_str(row.get("on_time")),
                "on_time_utc": utc_to_str(row.get("on_time")),
                "off_time_utc": utc_to_str(row.get("off_time")),
                "duration_s": float(row.get("duration_s", np.nan)) if pd.notna(row.get("duration_s", np.nan)) else np.nan,
                "source_x_m": source_x_m,
                "source_type": meta.get("source_type"),
                "operator": meta.get("operator"),
                "plate_type": meta.get("plate_type"),
                "n_blows": meta.get("n_blows"),
                "n_shots": meta.get("n_shots"),
                "survey": meta.get("survey"),
                "survey_type": meta.get("survey_type") or "nodal_detection",
                "shot_no": meta.get("shot_no"),
                "file_no": meta.get("file_no"),
                "matched_metadata_event_id": meta.get("matched_metadata_event_id"),
                "metadata_match_status": meta.get("metadata_match_status", "unmatched_detection"),
                "metadata_match_time_error_s": meta.get("metadata_match_time_error_s"),
                "detection_n_seed_ids": int(row.get("n_seed_ids", 0) or 0),
                "detection_n_stations": int(row.get("n_stations", 0) or 0) if "n_stations" in row else None,
                "detection_seed_ids": str(row.get("seed_ids", "")),
                "detection_stations": str(row.get("stations", "")),
                "snr_rms": float(row.get("snr_rms", np.nan)) if pd.notna(row.get("snr_rms", np.nan)) else np.nan,
                "n_receivers_extracted": n_receivers,
                "n_traces_extracted": n_traces,
                "status": status,
                "notes": meta.get("notes"),
            }
            pd.DataFrame([event_row]).to_sql("shot_events", conn, if_exists="append", index=False)

            if len(st_event) == 0:
                conn.commit()
                continue

            file_rows, trace_rows = write_component_products(st_event, label, event_id, gather_id, source_x_m)
            if file_rows:
                pd.DataFrame(file_rows).to_sql("shot_gather_files", conn, if_exists="append", index=False)
            if trace_rows:
                pd.DataFrame(trace_rows).to_sql("trace_index", conn, if_exists="append", index=False)

            picks_df = pick_event_stream(st_event, event_id, gather_id, source_x_m)
            if len(picks_df):
                pd.DataFrame(picks_df).to_sql("picks", conn, if_exists="append", index=False)
                picks_out = label_out / "tables" / "picks"
                picks_out.mkdir(parents=True, exist_ok=True)
                picks_df.to_csv(picks_out / f"{event_id}_picks.csv", index=False)

            conn.commit()
        except Exception as e:
            print("  FAILED:", e)
            log_error(conn, "event_processing", label=label, event_id=event_id, exc=e)
            continue

print("Done. Catalog:", CATALOG_DB)

## 9. Catalog inspection

Use SQL queries to inspect what was produced.

In [ ]:
def q(sql: str, params=None):
    return pd.read_sql(sql, conn, params=params or {})

print("processing_runs")
display(q("SELECT run_id, notebook_name, run_time_utc, input_sds_root, output_root FROM processing_runs ORDER BY run_time_utc DESC LIMIT 5"))

print("shot_events summary")
display(q("""
SELECT timewindow_label, status, COUNT(*) AS n_events,
       AVG(n_receivers_extracted) AS avg_receivers,
       MIN(n_receivers_extracted) AS min_receivers,
       MAX(n_receivers_extracted) AS max_receivers
FROM shot_events
WHERE run_id = :run_id
GROUP BY timewindow_label, status
ORDER BY timewindow_label, status
""", {"run_id": RUN_ID}))

print("files summary")
display(q("""
SELECT timewindow_label, file_type, component, COUNT(*) AS n_files
FROM shot_gather_files
WHERE run_id = :run_id
GROUP BY timewindow_label, file_type, component
ORDER BY timewindow_label, file_type, component
""", {"run_id": RUN_ID}))

print("pick summary")
display(q("""
SELECT timewindow_label, picker, pick_quality, COUNT(*) AS n
FROM picks p
JOIN shot_events e USING(event_id)
WHERE p.run_id = :run_id
GROUP BY timewindow_label, picker, pick_quality
ORDER BY timewindow_label, picker, pick_quality
""", {"run_id": RUN_ID}))

## 10. Optional CSV exports

SQLite is the source of truth, but these CSV snapshots are convenient for inspection, sharing, and debugging.

In [ ]:
if EXPORT_CSV_SNAPSHOTS:
    tables = [
        "processing_runs",
        "receiver_geometry",
        "shot_events",
        "shot_gather_files",
        "trace_index",
        "picks",
        "processing_errors",
    ]
    for table in tables:
        df = pd.read_sql(f"SELECT * FROM {table} WHERE run_id = ?" if table != "processing_runs" else "SELECT * FROM processing_runs WHERE run_id = ?", conn, params=(RUN_ID,))
        out = CSV_EXPORT_DIR / f"{RUN_ID}_{table}.csv"
        df.to_csv(out, index=False)
        print(table, len(df), "->", out)

## 11. Next notebooks

Recommended follow-ons:

- `91_stack_nodal_repeated_shots_by_metadata.ipynb`
- `92_extract_source_waveforms_and_spectra_from_catalog.ipynb`
- `93_compare_geode_streamer_nodal_common_shots.ipynb`
- `94_build_combined_supergathers.ipynb`

These should query `lbssp_shot_catalog.sqlite`, not reconstruct filenames manually.